# G17 — segmentation gallery (ground truth vs prediction, 30 test scans)
Picks the best finished model (highest single-model test Dice, plain prediction) from the attached **G17_loss_search_R3b** output and renders 30 rank-spaced test scans to `G17_segmentation_gallery.pdf`. CPU is enough.

In [ ]:
!pip -q install datasets

In [ ]:
%%writefile gallery_core.py
# G17 — segmentation gallery: ground-truth area vs predicted area for one model on the 266-scan test set.
import os
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Patch

GT_C, PR_C = "#2ecc71", "#ff4757"


def pick_cases(df, n=30):
    """Evenly spaced ranks over the whole Dice distribution → a representative, unbiased sample
    (best, typical and worst cases in their true proportions)."""
    s = df.sort_values("dice", ascending=False).reset_index()
    idx = np.unique(np.round(np.linspace(0, len(s) - 1, n)).astype(int))
    return s.iloc[idx]


def _shrink(img, *masks, side=520):
    h, w = img.shape
    f = min(1.0, side / max(h, w))
    if f >= 1.0:
        return (img,) + masks
    sz = (int(w * f), int(h * f))
    return (cv2.resize(img, sz, interpolation=cv2.INTER_AREA),) + tuple(
        cv2.resize(m.astype(np.uint8), sz, interpolation=cv2.INTER_NEAREST) > 0 for m in masks)


def overlay(ax, img, mask, colour, title, ref=None):
    ax.imshow(img, cmap="gray", interpolation="lanczos")
    if mask.any():
        rgba = np.zeros(mask.shape + (4,), np.float32)
        rgba[mask] = matplotlib.colors.to_rgba(colour, 0.38)
        ax.imshow(rgba, interpolation="nearest")
        ax.contour(mask, levels=[0.5], colors=colour, linewidths=1.3)
    if ref is not None and ref.any():   # thin dashed ground-truth outline for comparison
        ax.contour(ref, levels=[0.5], colors=GT_C, linewidths=0.8, linestyles="--")
    ax.set_title(title, fontsize=7.5, loc="left")
    ax.axis("off")


def build_gallery(runs_root, run, label, split_test, out_pdf, n=30, variant="plain", per_page=6):
    d = Path(runs_root) / run
    df = pd.read_csv(d / f"test_{variant}.csv").set_index("image_id")
    probs = np.load(d / "test_probs.npz")["tta" if "tta" in variant else "plain"]
    ids = list(pd.read_csv(d / "test_plain.csv").image_id)
    pos = {c: i for i, c in enumerate(ids)}
    native = {r["image_id"]: (np.asarray(r["raw_image"].convert("L")), np.asarray(r["mask"].convert("L")) > 127)
              for r in split_test}
    cases = pick_cases(df, n)
    pdf = PdfPages(out_pdf)
    pngs = []

    # ---- cover page
    fig = plt.figure(figsize=(11.69, 8.27))
    fig.text(0.06, 0.88, "G17 — Lesion segmentation: ground truth vs prediction", fontsize=20, weight="bold")
    fig.text(0.06, 0.83, f"Model: {label}", fontsize=12)
    m = df
    rows = [("Test scans", f"{len(m)}"),
            ("Mean Dice", f"{m.dice.mean()*100:.2f} %"),
            ("Mean IoU", f"{m.jaccard.mean()*100:.2f} %"),
            ("Mean 95% Hausdorff", f"{m.hd95.mean():.2f} px"),
            ("Mean surface distance", f"{m.asd.mean():.2f} px"),
            ("Scans with Dice < 50 %", f"{(m.dice < 0.5).mean()*100:.1f} %")]
    for k, (a, b) in enumerate(rows):
        fig.text(0.08, 0.74 - k * 0.045, a, fontsize=12)
        fig.text(0.36, 0.74 - k * 0.045, b, fontsize=12, weight="bold")
    coh = m.groupby("dataset").dice.agg(["size", "mean"]) if "dataset" in m else None
    if coh is not None:
        fig.text(0.56, 0.74, "By centre", fontsize=12, weight="bold")
        for k, (c, r) in enumerate(coh.iterrows()):
            fig.text(0.56, 0.70 - k * 0.04, f"{c}  (n={int(r['size'])})", fontsize=11)
            fig.text(0.78, 0.70 - k * 0.04, f"{r['mean']*100:.2f} % Dice", fontsize=11)
    fig.text(0.06, 0.38,
             f"The {len(cases)} scans on the following pages were chosen by rank, evenly spaced across the full\n"
             "Dice distribution of the test set (from the best to the worst case), so they show typical behaviour\n"
             "rather than hand-picked successes. Each scan is shown at full frame in its native resolution.\n\n"
             "Left panel: ground-truth lesion area (green).   Right panel: predicted lesion area (red),\n"
             "with the ground-truth outline repeated as a thin dashed green line for comparison.\n"
             "Areas are in pixels of the original image; Dice measures the overlap of the two areas.\n"
             "Threshold 0.5, single forward pass (no post-processing).", fontsize=11, va="top")
    fig.legend(handles=[Patch(color=GT_C, alpha=0.6, label="Ground truth"), Patch(color=PR_C, alpha=0.6, label="Prediction")],
               loc="lower left", bbox_to_anchor=(0.06, 0.08), ncol=2, fontsize=11, frameon=False)
    pdf.savefig(fig); plt.close(fig)

    # ---- case pages: 3 rows × (GT | Pred) × 2 cases per row
    case_list = list(cases.itertuples())
    for p0 in range(0, len(case_list), per_page):
        chunk = case_list[p0:p0 + per_page]
        fig, axes = plt.subplots(3, 4, figsize=(11.69, 8.27))
        for k in range(per_page):
            r, c = divmod(k, 2)
            a_gt, a_pr = axes[r, 2 * c], axes[r, 2 * c + 1]
            if k >= len(chunk):
                a_gt.axis("off"); a_pr.axis("off"); continue
            row = chunk[k]
            cid = row.image_id
            img, gt = native[cid]
            pr = cv2.resize(probs[pos[cid]].astype(np.float32) / 255.0, (gt.shape[1], gt.shape[0]),
                            interpolation=cv2.INTER_LINEAR) >= 0.5
            n_case = p0 + k + 1
            gt_px, pr_px = int(gt.sum()), int(pr.sum())
            img, gt, pr = _shrink(img, gt, pr)
            overlay(a_gt, img, gt, GT_C, f"#{n_case}  {row.dataset} — {cid[:28]}\nGround truth: {gt_px:,} px")
            overlay(a_pr, img, pr, PR_C, f"Dice {row.dice*100:.1f} %  ·  95HD {row.hd95:.1f} px\nPrediction: {pr_px:,} px", ref=gt)
        fig.suptitle(f"{label} — test scans {p0+1}–{p0+len(chunk)} of {len(case_list)} (sorted best → worst)",
                     fontsize=11, x=0.01, ha="left")
        fig.legend(handles=[Patch(color=GT_C, alpha=0.6, label="Ground truth"), Patch(color=PR_C, alpha=0.6, label="Prediction")],
                   loc="lower right", ncol=2, fontsize=9, frameon=False)
        fig.tight_layout(rect=(0, 0.03, 1, 0.95), h_pad=2.6, w_pad=0.6)
        png = str(Path(out_pdf).with_suffix("")) + f"_p{p0 // per_page + 1}.png"
        fig.savefig(png, dpi=130); pngs.append(png)
        pdf.savefig(fig, dpi=150); plt.close(fig)
    pdf.close()
    return pngs


In [ ]:
import glob, shutil, sys, os, pandas as pd
src = sorted(glob.glob('/kaggle/input/**/g17core.py', recursive=True))[0]; shutil.copy(src, 'g17core.py')
sys.path.insert(0, '.'); import g17core as g, gallery_core as G
root = sorted(d for d in glob.glob('/kaggle/input/**/runs', recursive=True) if os.path.isdir(d))[0]
rows = []
for r in sorted(os.listdir(root)):
    f = os.path.join(root, r, 'test_plain.csv')
    if os.path.exists(f) and os.path.exists(os.path.join(root, r, 'test_probs.npz')) and not r.startswith('ENS'):
        d = pd.read_csv(f); v = os.path.join(root, r, 'summary.csv')
        val = pd.read_csv(v) if os.path.exists(v) else None
        vd = val[(val.split == 'val') & (val.variant == 'plain') & (val.cohort == 'Overall')].dice.max() if val is not None and 'split' in val else float('nan')
        rows.append((r, d.dice.mean() * 100, vd))
board = pd.DataFrame(rows, columns=['run', 'test_dice', 'val_dice']).sort_values('test_dice', ascending=False)
print(board.head(10).round(2).to_string(index=False))
BEST = board.iloc[0].run
LABELS = {'R3_upp_r34_ft55': 'U-Net++ (ResNet-34, ImageNet-pretrained) + Focal Tversky α0.5/β0.5',
          'R3_upp_r34_dice': 'U-Net++ (ResNet-34, ImageNet-pretrained) + BCE + Dice',
          'R3_upp_r34_ft55_fpd1': 'U-Net++ (ResNet-34, ImageNet-pretrained) + Focal Tversky + false-positive penalty'}
print('BEST =', BEST)
split = g.load_splits()
pngs = G.build_gallery(root, BEST, LABELS.get(BEST, BEST), split['test'], 'G17_segmentation_gallery.pdf', n=30)

In [ ]:
from IPython.display import Image, display
for p in pngs: display(Image(p))